In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install emoji==0.6.0


In [ ]:
!pip install pytorch-lightning


In [ ]:
!pip install contractions

In [ ]:
import os
import csv
import random
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from typing import Dict, Any
from transformers import AutoTokenizer, AutoModel
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from transformers import get_cosine_schedule_with_warmup
import torch.nn.functional as F
import random, numpy as np, torch
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="emoji is not installed*")
torch.set_float32_matmul_precision("high")
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer,ENGLISH_STOP_WORDS
from collections import Counter,defaultdict
import contractions
import re
from sklearn.metrics import classification_report
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
def expand_and_clean(text: str) -> str:
    expanded_text = contractions.fix(text)
    expanded_text = re.sub(r"[^a-zA-Z\s]", " ", expanded_text)
    expanded_text = re.sub(r"\s+", " ", expanded_text).strip().lower()

    return expanded_text

def compute_label_weighted_tfidf(train_dataset, tfidf_vectorizer, top_k=200):
    texts = [expand_and_clean(item["text"]) for item in train_dataset.samples]
    labels = np.array([item["label"] for item in train_dataset.samples])

    X = tfidf_vectorizer.transform(texts)
    feature_names = tfidf_vectorizer.get_feature_names_out()

    keyword_scores = {}
    for i, word in enumerate(feature_names):
        score = X[labels==1, i].mean() - X[labels==0, i].mean()
        keyword_scores[word] = score

    # top_k
    top_keywords = sorted(keyword_scores.items(), key=lambda x: abs(x[1]), reverse=True)[:top_k]
    return top_keywords

class TranscriptFolderDataset(Dataset):
    def __init__(
        self,
        split_dir,
        tokenizer,
        max_length=512,
        text_column="text",
        label_column="Depression_label"
    ):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []

        labels_path = os.path.join(split_dir, "labels.csv")
        transcripts_dir = os.path.join(split_dir, "samples")

        assert os.path.isfile(labels_path), f"Missing {labels_path}"
        assert os.path.isdir(transcripts_dir), f"Missing {transcripts_dir}"

        label_map = {}
        with open(labels_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                pid = row["Participant"].strip()
                label_map[pid] = int(row["Depression_label"])

        for pid, label in label_map.items():
            transcript_file = os.path.join(transcripts_dir, f"{pid}_P_TRANSCRIPT.csv")
            if not os.path.isfile(transcript_file):
                continue

            utterances = []
            with open(transcript_file, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    text = (row.get(text_column) or "").strip()
                    if text:
                        utterances.append(text)

            if len(utterances) == 0:
                continue

            full_text = " ".join(utterances)

            self.samples.append({
                "text": full_text,
                "label": label
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        enc = self.tokenizer(
            item["text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }


class EBTClassifier(pl.LightningModule):
    def __init__(self, hparams: Dict[str, Any], tokenizer=None, tfidf_vectorizer=None):
        super().__init__()
        self.save_hyperparameters(hparams)
        self.hp = hparams
        self.num_labels = hparams.get("num_labels", 2)
        self.encoder_name = hparams.get("encoder_name", "roberta-base")
        self.tokenizer = AutoTokenizer.from_pretrained(self.encoder_name)
        counts = torch.tensor([103, 29], dtype=torch.float)
        priors = counts / counts.sum()
        log_priors = torch.log(priors + 1e-8)
        self.prior_alpha = 0.25
        self.register_buffer("log_priors", log_priors)
        self.test_preds = []
        self.test_labels = []
        self.train_loss_epoch = []
        self.val_loss_epoch = []
        self.test_probs = []
        self.test_attn = []
        self.test_input_ids = []
        # Encoder
        self.encoder = AutoModel.from_pretrained(self.encoder_name)
        hid = self.encoder.config.hidden_size
        self.self_attn = nn.MultiheadAttention(
            embed_dim=hid,
            num_heads=8,
            batch_first=True
        )
        self.gate = nn.Linear(hid, hid)
        self.attn = nn.Linear(hid, 1)
        self.norm = nn.LayerNorm(hid)
        self.dropout = nn.Dropout(0.2)
        self.energy_mlp = nn.Sequential(
            nn.Linear(hid + self.num_labels, 512),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Linear(256, 1),
        )

        # Freeze embeddings
        for param in self.encoder.embeddings.parameters():
            param.requires_grad = False

        # Freeze first 6 transformer layers
        for layer in self.encoder.encoder.layer[:6]:
            for param in layer.parameters():
                param.requires_grad = False
        # Optimization hyperparams
        self.y_steps = int(hparams.get("mcmc_num_steps", 2))
        self.y_step_size = float(hparams.get("mcmc_step_size", 0.5))
        self.langevin_sigma = float(hparams.get("langevin_sigma", 0.01))
        self.y_learnable_step = bool(hparams.get("mcmc_step_size_learnable", False))
        if self.y_learnable_step:
            self._alpha_raw = nn.Parameter(torch.tensor(self.y_step_size, dtype=torch.float32))
        else:
            self.register_buffer("_alpha_buf", torch.tensor(self.y_step_size, dtype=torch.float32))
        self.use_replay = hparams.get("use_replay_buffer", False)
        if self.use_replay:
            self.replay_buffer = []
            self.replay_size = int(hparams.get("replay_size", 1000))

        self.system_mode = hparams.get("system_mode", "S2")
        # Loss for classification
        class_weights = torch.tensor([1.0, 103/29], dtype=torch.float)
        self.register_buffer("class_weights", class_weights)
        self.classification_loss = nn.CrossEntropyLoss(weight=class_weights)

        self.val_loader_for_eval = None
    def on_fit_start(self):
        import time, torch
        torch.cuda.synchronize()
        self.train_start_time = time.time()

    def on_fit_end(self):
        import time, torch
        torch.cuda.synchronize()

        total_time = time.time() - self.train_start_time

        print(f"Total training time: {total_time:.2f} sec")

        self.logger.experiment.add_scalar("total_training_time", total_time)
    def get_alpha(self):
        if getattr(self, "y_learnable_step", False):
            return torch.nn.functional.softplus(self._alpha_raw)
        return self._alpha_buf

    def encode_text(self, input_ids, attention_mask, return_attn=False):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        token_embeddings = out.last_hidden_state

        attn_tokens, _ = self.self_attn(
            token_embeddings,
            token_embeddings,
            token_embeddings,
            key_padding_mask=~attention_mask.bool()
        )

        attn_tokens = self.norm(attn_tokens + token_embeddings)

        gate = torch.sigmoid(self.gate(attn_tokens))
        gated_tokens = attn_tokens * gate

        # Attention pooling
        scores = self.attn(gated_tokens).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, -1e9)

        weights = torch.softmax(scores, dim=1)

        context_embed = (weights.unsqueeze(-1) * gated_tokens).sum(dim=1)
        context_embed = self.dropout(context_embed)
        if return_attn:
            return context_embed, weights
        return context_embed
    def energy(self, context_embed, y_logits):
        # y_logits: (B, num_labels)
        combined = torch.cat([context_embed, y_logits], dim=-1)
        energy_val = self.energy_mlp(combined)  # (B, 1)
        return energy_val
    def optimize_y(self, context_embed, y_init=None, num_steps=None, tfidf_bias=None):
        B = context_embed.size(0)
        device = context_embed.device
        if num_steps is None:
            num_steps = self.y_steps
        if self.training:
            num_steps = int(max(1, num_steps + random.choice([-1, 0, 1])))

        if y_init is None:
            y = torch.randn((B, self.num_labels), device=device, dtype=torch.float32, requires_grad=True)
        else:
            y = y_init.clone().to(device).float().requires_grad_(True)

        create_graph_flag = bool(getattr(self, "y_learnable_step", False) and self.training)

        for step_idx in range(num_steps):
            # compute energy and gradient wrt y
            with torch.enable_grad():
                e = self.energy(context_embed, y)   # (B,1)
                if tfidf_bias is not None:
                    e = e + self.hp.get("tfidf_energy_coef", 0.01) * tfidf_bias
                e_sum = e.sum()
                grad_y_tuple = torch.autograd.grad(
                    e_sum,
                    y,
                    create_graph=create_graph_flag,
                    retain_graph=create_graph_flag,
                    allow_unused=False
                )
            grad_y = grad_y_tuple[0] if isinstance(grad_y_tuple, tuple) else grad_y_tuple
            if grad_y is None:
                raise RuntimeError("optimize_y: grad_y is None")

            # normalize gradient
            grad_norm = grad_y.norm(dim=-1, keepdim=True).clamp_min(1e-6)
            grad_y = grad_y / (grad_norm + 1e-8)

            step = self.get_alpha()
            if hasattr(step, "device") and step.device != device:
                step = step.to(device)
            step = step.float()

            if self.training:
                rand_multiplier = torch.empty(1, device=device).uniform_(0.8, 1.2)
                step = step * rand_multiplier

            y = y - step * grad_y

            if self.training and getattr(self, "langevin_sigma", 0.0) > 0:
                y = y + self.langevin_sigma * torch.randn_like(y)

            y = y.requires_grad_(True)

        return y
    def tfidf_sentence_prior(self, input_ids):
        if not hasattr(self, "pos_keyword_scores"):
            return torch.zeros((input_ids.size(0), 1), device=input_ids.device)

        priors = []
        for i in range(input_ids.size(0)):
            tokens = self.tokenizer.convert_ids_to_tokens(input_ids[i])
            s, cnt = 0.0, 0
            for tok in tokens:
                tok = tok.replace("Ġ", "").replace("##", "").lower()
                tok = re.sub(r"[^a-z0-9]", "", tok)
                if tok in self.pos_keyword_scores:
                    s += self.pos_keyword_scores[tok]
                    cnt += 1
            priors.append(s / max(cnt, 1))

        return torch.tensor(priors, device=input_ids.device).unsqueeze(1)
    def save_keywords_to_csv(self,csv_path="/content/drive/MyDrive/EBT/keywords_depression.csv"):
        if not hasattr(self, "pos_keyword_list") or not hasattr(self, "pos_keyword_scores"):
            print("No pos keywords present on model. Nothing saved.")
            return

        words = list(self.pos_keyword_list)
        scores = [self.pos_keyword_scores.get(w, 0.0) for w in words]

        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["keyword", "score"])
            for w, s in zip(words, scores):
                writer.writerow([w, s])

        print(f"Saved {len(words)} used keywords → {csv_path}")

    def compute_metrics(self, batch):
        input_ids, attention_mask, labels = batch["input_ids"], batch["attention_mask"], batch["labels"]
        input_ids = input_ids.to(self.device)
        attention_mask = attention_mask.to(self.device)
        labels = labels.to(self.device).long()

        if self.training:
            context_embed = self.encode_text(input_ids, attention_mask)
        else:
            with torch.no_grad():
                context_embed = self.encode_text(input_ids, attention_mask)
            context_embed = context_embed.detach()

        # Optimize y
        steps = 1 if self.system_mode == "S1" else self.y_steps
         # (1) first-pass y (no TF-IDF)
        y_init = self.optimize_y(context_embed, num_steps=steps)

        # (2) uncertainty
        with torch.no_grad():
            probs = torch.softmax(y_init, dim=-1)[:, 1]
            uncertainty = 1.0 - torch.abs(probs - 0.5) * 2
            uncertainty = uncertainty.clamp(0.0, 1.0)

        # (3) TF-IDF prior
        tfidf_bias = self.tfidf_sentence_prior(input_ids)

        if self.training:
            tfidf_bias = tfidf_bias * 0.3
        else:
            tfidf_bias = tfidf_bias * uncertainty.unsqueeze(1)

        # (4) second-pass y with lexical prior
        y_optimized = self.optimize_y(
            context_embed,
            y_init=y_init.detach(),
            num_steps=steps,
            tfidf_bias=tfidf_bias
        )
        if not self.training:
            y_optimized = y_optimized + self.prior_alpha * self.log_priors

        # Compute classification loss
        loss = self.classification_loss(y_optimized, labels)

        # Energy regularization (keep gradient!)
        energy_val = self.energy(context_embed, y_optimized)
        energy_reg = energy_val.mean()
        total_loss = loss + self.hp.get("energy_reg_coef", 0.01) * energy_reg

        with torch.no_grad():
            threshold = 0.4

            probs = torch.softmax(y_optimized, dim=-1)[:, 1]
            preds = (probs >= threshold).long()
            acc = (preds == labels).float().mean()

        return total_loss, acc, loss.detach(), energy_reg.detach(), preds.detach(),labels.detach(),probs.detach()


    def on_train_epoch_end(self):
        self.train_loss_epoch.append(self.trainer.callback_metrics["train_loss"].item())

    def on_validation_epoch_end(self):
        if "val_loss" in self.trainer.callback_metrics:
            self.val_loss_epoch.append(self.trainer.callback_metrics["val_loss"].item())
    def training_step(self, batch, batch_idx):
        total_loss, acc, ce_loss, energy_reg, preds, labels, _ = self.compute_metrics(batch)
        self.encoder.train()

        preds_np = preds.cpu().numpy()
        labels_np = labels.cpu().numpy()
        precision = precision_score(labels_np, preds_np, average="macro", zero_division=0)
        recall = recall_score(labels_np, preds_np, average="macro", zero_division=0)
        f1 = f1_score(labels_np, preds_np, average="macro", zero_division=0)

        # log (step-level; Lightning will also aggregate if you set on_epoch=True)
        self.log("train_loss", total_loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_precision_macro", precision, prog_bar=False, on_step=True, on_epoch=True)
        self.log("train_recall_macro", recall, prog_bar=False, on_step=True, on_epoch=True)
        self.log("train_f1_macro", f1, prog_bar=True, on_step=True, on_epoch=True)

        return total_loss

    def validation_step(self, batch, batch_idx):
        total_loss, acc, ce_loss, energy_reg, preds, labels, _ = self.compute_metrics(batch)

        preds_np = preds.cpu().numpy()
        labels_np = labels.cpu().numpy()
        balanced_acc = balanced_accuracy_score(labels_np, preds_np)

        precision = precision_score(labels_np, preds_np, average="macro", zero_division=0)
        recall = recall_score(labels_np, preds_np, average="macro", zero_division=0)
        f1 = f1_score(labels_np, preds_np, average="macro", zero_division=0)

        self.log("val_loss", total_loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_precision_macro", precision, prog_bar=False, on_step=False, on_epoch=True)
        self.log("val_recall_macro", recall, prog_bar=False, on_step=False, on_epoch=True)
        self.log("val_f1_macro", f1, prog_bar=True, on_step=False, on_epoch=True)

        return {"val_loss": total_loss, "val_acc": acc, "val_precision_macro": precision, "val_recall_macro": recall, "val_f1_macro": f1}
    def on_test_epoch_end(self):
        y_true = torch.cat(self.test_labels).numpy()
        y_pred = torch.cat(self.test_preds).numpy()

        cm = confusion_matrix(y_true, y_pred)

        fig, ax = plt.subplots(figsize=(4, 4))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Non-Depressed", "Depressed"],
            yticklabels=["Non-Depressed", "Depressed"],
            ax=ax
        )
        ax.set_xlabel("Predicted Label")
        ax.set_ylabel("True Label")
        ax.set_title("Confusion Matrix (Dataset 3)")
        version = self.logger.version
        save_dir = f"/content/drive/MyDrive/EBT/figures/TFIDF_D3/version_{version}"
        os.makedirs(save_dir, exist_ok=True)
        fig_path = os.path.join(save_dir, "CM_D3M1_EBT_Baseline.png")

        fig.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        report_dict = classification_report(
            y_true,
            y_pred,
            target_names=["Non-Depressed", "Depressed"],
            output_dict=True
        )

        df = pd.DataFrame(report_dict).transpose()
        df.to_csv(os.path.join(save_dir, "classification_report.csv"))

        # Attention Visualization
        sample_weights = self.test_attn[0][0].numpy()
        sample_ids = self.test_input_ids[0][0].numpy()
        weights = sample_weights
        tokens = self.tokenizer.convert_ids_to_tokens(sample_ids)
        tokens = [t.replace("Ġ", "") for t in tokens]
        weights = sample_weights
        stopwords = {"the", "a", "in", "on", "and", "of", "to", "is", "are", "that", "have", "but"}

        filtered = [
            (t, w) for t, w in zip(tokens, weights)
            if t.lower() not in stopwords and len(t) > 2
        ]

        tokens, weights = zip(*filtered)
        top_k = 15
        indices = np.argsort(weights)[-top_k:]

        top_tokens = [tokens[i] for i in indices]
        top_weights = [weights[i] for i in indices]
        plt.figure(figsize=(10, 2))
        sns.heatmap([top_weights], cmap="Reds", xticklabels=top_tokens)

        plt.xticks(rotation=45)
        plt.title("Top Attention Tokens")
        plt.tight_layout()

        plt.savefig(os.path.join(save_dir, "attention_topk_heatmap.png"), dpi=300)
        plt.close()

        print(f"Saved report & attention to {save_dir}")
        self.test_preds.clear()
        self.test_labels.clear()
        self.test_attn.clear()


    def test_step(self, batch, batch_idx):
        total_loss, acc, ce_loss, energy_reg, preds, labels, _ = self.compute_metrics(batch)

        self.test_preds.append(preds.detach().cpu())
        self.test_labels.append(labels.detach().cpu())

        preds_np = preds.cpu().numpy()
        labels_np = labels.cpu().numpy()

        precision = precision_score(labels_np, preds_np, average="macro", zero_division=0)
        recall = recall_score(labels_np, preds_np, average="macro", zero_division=0)
        f1 = f1_score(labels_np, preds_np, average="macro", zero_division=0)

        self.log("test_loss", total_loss, prog_bar=True, on_epoch=True)
        self.log("test_acc", acc, prog_bar=True, on_epoch=True)
        self.log("test_precision_macro", precision, on_epoch=True)
        self.log("test_recall_macro", recall, on_epoch=True)
        self.log("test_f1_macro", f1, prog_bar=True, on_epoch=True)
        input_ids = batch["input_ids"]

        attention_mask = batch["attention_mask"]

        context_embed, attn_weights = self.encode_text(
            input_ids, attention_mask, return_attn=True
        )

        total_loss, acc, ce_loss, energy_reg, preds, labels, _ = self.compute_metrics(batch)
        self.test_attn.append(attn_weights.detach().cpu())
        self.test_input_ids.append(input_ids.detach().cpu())
        return total_loss


    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=3e-5, weight_decay=0.01)

        # warmup + cosine
        num_training_steps = self.trainer.estimated_stepping_batches
        num_warmup_steps = int(0.1 * num_training_steps)

        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
                "frequency": 1
            }
        }


if __name__ == "__main__":
    hparams = {
        "tokenizer_name": "roberta-base",
        "encoder_name": "roberta-base",
        "num_labels": 2,
        "energy_hidden": 512,
        "mcmc_num_steps": 2,
        "mcmc_step_size": 0.05,
        "mcmc_step_size_learnable": True,
        "langevin_sigma": 0.01,
        "use_replay_buffer": False,
        "energy_reg_coef": 0.02,
        "system_mode": "S2",
        "nfe_steps_to_eval": [1, 2, 4, 6],
        "nfe_eval_batches_per_step": 8,
        "tfidf_energy_coef": 0.01,
    }

    tokenizer = AutoTokenizer.from_pretrained(hparams["encoder_name"])
    train_dir = "/content/drive/MyDrive/EBT/data/Dataset 3/train"
    val_dir   = "/content/drive/MyDrive/EBT/data/Dataset 3/val"
    test_dir  = "/content/drive/MyDrive/EBT/data/Dataset 3/test"

    train_dataset = TranscriptFolderDataset(train_dir, tokenizer)
    val_dataset   = TranscriptFolderDataset(val_dir, tokenizer)
    test_dataset  = TranscriptFolderDataset(test_dir, tokenizer)
    labels = [s["label"] for s in train_dataset.samples]

    class_counts = np.bincount(labels)
    print("Train class counts:", class_counts)

    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[l] for l in labels]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    # Clean texts and tokenize
    train_texts_clean = [expand_and_clean(item["text"]) for item in train_dataset.samples]

    # Count token frequencies
    all_tokens = []
    for text in train_texts_clean:
        all_tokens.extend(text.split())
    token_counts = Counter(all_tokens)

    # Top 50 high-frequency tokens
    high_freq_tokens = [tok for tok, _ in token_counts.most_common(50)]

    # Subword noise from tokenizer
    subword_noise = [tok for tok in tokenizer.get_vocab().keys() if tok.startswith("##") or len(tok) <= 2]
    TRANSCRIPT_FILLERS = [
    "um","uh","yeah","yes","okay","ok",
    "right","well","like","hmm"
    ]
    INTERVIEW_WORDS = [
    "therapist",
    "interviewer",
    "participant",
    "son",
    "daughter",
    "mother",
    "father"
]
    STOPWORDS = set(ENGLISH_STOP_WORDS) | set(high_freq_tokens) | set(subword_noise) | set(["wa","ve","im","ha","id","u","ur","lol","omg","http","https","www","amp","com"])| set(TRANSCRIPT_FILLERS)| set(INTERVIEW_WORDS)
    STOPWORDS = [str(s) for s in STOPWORDS]

    tfidf_vectorizer = TfidfVectorizer(
        max_features=5000,
        stop_words=STOPWORDS,
        token_pattern=r"(?u)\b[a-z]{3,}\b",
        ngram_range=(1,2),
        min_df=2
    )
    tfidf_vectorizer.fit(train_texts_clean)
    top_keywords = compute_label_weighted_tfidf(train_dataset, tfidf_vectorizer, top_k=200)

    # split pos / neg
    min_score_threshold = 0.005
    K = 100
    pos_keywords = [(w,s) for w,s in top_keywords if s > 0.0]
    # keep only positive keywords above threshold
    pos_filtered = [(w, s) for w, s in pos_keywords if s >= min_score_threshold]
    if len(pos_filtered) > 0:
        pos_words, pos_scores = zip(*pos_filtered)
        pos_scores = np.array(pos_scores, dtype=float)
        pos_scores_norm = pos_scores / (pos_scores.max() + 1e-9)
    else:
        pos_words, pos_scores_norm = [], []

    model = EBTClassifier(hparams, tokenizer=tokenizer)
    model.tokenizer = tokenizer
    model.pos_keyword_list = list(pos_words)
    model.pos_keyword_scores = {w: float(s) for w, s in zip(pos_words, pos_scores_norm)}
    print(f"Selected {len(pos_words)} positive keywords (top-{K})")

    model.STOPWORDS = STOPWORDS

    model.save_keywords_to_csv("/content/drive/MyDrive/EBT/D3_keywords_depression.csv")
    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        sampler=sampler,
        num_workers=2
    )
    for batch in train_loader:
        print(batch["labels"])
        break


    val_loader   = DataLoader(val_dataset, batch_size=8, num_workers=2)
    test_loader  = DataLoader(test_dataset, batch_size=8, num_workers=2)


    model.val_loader_for_eval = val_loader
    tlogger = TensorBoardLogger("logs", name="TFIDF_D3", version=None)
    early_stop = EarlyStopping(monitor="val_f1_macro", patience=5, mode="max")
    ckpt_callback = ModelCheckpoint(
        dirpath="ckpt",
        monitor="val_f1_macro",
        mode="max",
        save_top_k=1,
        save_last=False,
        filename="best-{epoch:02d}-{val_f1_macro:.4f}"
    )

    trainer = pl.Trainer(
        max_epochs=20,
        accelerator="gpu",
        devices=1,
        logger=tlogger,
        callbacks=[early_stop, ckpt_callback],
        precision=32,
        gradient_clip_val=1.0,
        inference_mode=False,
        num_sanity_val_steps=0,
    )

    trainer.fit(model, train_loader, val_loader)
    best_ckpt_path = ckpt_callback.best_model_path
    print(f"Best checkpoint: {best_ckpt_path}")

    trainer.test(model=model, dataloaders=test_loader, ckpt_path=best_ckpt_path)


Train class counts: [103  29]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Selected 138 positive keywords (top-100)
Saved 138 used keywords → /content/drive/MyDrive/EBT/D3_keywords_depression.csv
tensor([0, 0, 1, 0, 0, 0, 1, 1])


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.


...

Best checkpoint: /content/ckpt/best-epoch=16-val_f1_macro=0.7952.ckpt


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from the checkpoint at /content/ckpt/best-epoch=16-val_f1_macro=0.7952.ckpt


Output()

Saved report & attention to /content/drive/MyDrive/EBT/figures/TFIDF_D3/version_1

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.8275862336158752     │
│       test_f1_macro       │    0.7122142314910889     │
│         test_loss         │     1.018450379371643     │
│   test_precision_macro    │    0.7126436829566956     │
│     test_recall_macro     │    0.7733989953994751     │
└───────────────────────────┴───────────────────────────┘

In [ ]:
import shutil
save_log_path = f"/content/drive/MyDrive/EBT/LOG"
save_ckpt_path = f"/content/drive/MyDrive/EBT/LOG"
shutil.copytree("logs", save_log_path, dirs_exist_ok=True)
shutil.copytree("ckpt", save_ckpt_path, dirs_exist_ok=True)
print("Saved logs to:", save_log_path)
print("Saved ckpt to:", save_ckpt_path)

Saved logs to: /content/drive/MyDrive/EBT/LOG
Saved ckpt to: /content/drive/MyDrive/EBT/LOG


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/EBT/LOG/TFIDF_D3"